In [ ]:
!pip install stable-baselines3
!pip install aquacrop
!git clone https://github.com/aquacropos/aquacrop-gym.git
%cd aquacrop-gym
!pip install tensorflow




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.0 MB/s eta 0:00:00
Cloning into 'aquacrop-gym'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 129 (delta 26), reused 117 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 18.46 MiB | 22.75 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/aquacrop-gym


In [ ]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import numpy as np
import random
import aquacropgym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import aquacrop

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather, get_filepath
weather_file_path = get_filepath('tunis_climate.txt')


In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather, get_filepath


class AquaCropIrrigationEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.max_season_water = 700.0  # лимит воды за сезон в мм
        self.weather = prepare_weather(get_filepath("tunis_climate.txt"))
        self.weather["Precipitation"] = 0.0
        self.start_date = "1979-10-01"
        self.end_date = "1980-05-30"
        self.max_days = 180

        self.action_space = spaces.Box(
            low=np.array([0.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
          low=np.zeros(8, dtype=np.float32),
          high=np.ones(8, dtype=np.float32),
          dtype=np.float32
        )

        self.day = 0
        self.total_water = 0.0
        self.irrigation_schedule = None
        self.model = None

    def _make_schedule(self):
      return pd.DataFrame({
          "Date": pd.date_range(self.start_date, self.end_date, freq="D"),
          "Depth": np.zeros(self.max_days, dtype=float)
      })

    def _build_model(self):
      irrigation = IrrigationManagement(
          irrigation_method=5,
          depth=0.0,
          MaxIrr=30.0
      )

      model = AquaCropModel(
        sim_start_time="1979/10/01",
        sim_end_time="1980/05/30",
        weather_df=self.weather,
        soil=Soil("SandyLoam"),
        crop=Crop("Wheat", planting_date="10/01"),
        initial_water_content=InitialWaterContent(value=["WP"]),
        irrigation_management=irrigation
      )

      return model

    def reset(self, seed=None, options=None):
      super().reset(seed=seed)

      self.day = 0
      self.total_water = 0
      self.total_yield = 0
      self.total_yield = 0.0
      self.model = self._build_model()

      # Важно: просто инициализируем модель, не пересоздаём её потом каждый день
      self.model._initialize()

      return self._get_obs(), {}

    def step(self, action):
      raw_action = float(np.asarray(action).reshape(-1)[0])
      raw_action = float(np.clip(raw_action, 0.0, 1.0))

      requested_irrigation = raw_action * 30.0

      remaining_water = max(self.max_season_water - self.total_water, 0.0)

      irrigation = min(requested_irrigation, remaining_water)

      prev_biomass = float(self.model._init_cond.biomass)
      prev_irr_cum = float(self.model._init_cond.irr_cum)
      prev_th = float(self.model._init_cond.th[0])
      self.model._param_struct.IrrMngt.depth = irrigation

      # ВАЖНО: без num_steps=1
      self.model.run_model(initialize_model=False)

      obs = self._get_obs()

      biomass_now = float(self.model._init_cond.biomass)
      irr_cum_now = float(self.model._init_cond.irr_cum)

      biomass_gain = biomass_now - prev_biomass
      actual_irrigation_today = irr_cum_now - prev_irr_cum
      self.total_water += actual_irrigation_today

      th1 = float(self.model._init_cond.th[0])

      reward = 0.0

      # Награда за рост
      reward += biomass_gain

      # Если ДО действия почва была сухая и агент полил — награда
      if prev_th < 0.23 and irrigation >= 5.0:
          reward += 5.0

      # Хорошая влажность после действия
      if 0.25 <= th1 <= 0.36:
          reward += 8.0

      # Слишком сухо после действия
      if th1 < 0.22:
          reward -= 20.0

      # Перелив
      if th1 > 0.40:
          reward -= 5.0

      # Маленький штраф за воду
      reward -= irrigation*0.9

      terminated = self.day >= self.max_days - 1
      truncated = False
      info = {}
      info = {}

      terminated = self.day >= self.max_days - 1
      truncated = False

      if terminated:
          self.model.run_model(till_termination=True, initialize_model=False)
          results = self.model.get_simulation_results()

          self.final_yield = float(results["Dry yield (tonne/ha)"].iloc[-1])
          self.total_yield = self.final_yield

          reward += self.final_yield * 30

          if self.final_yield <= 0:
              reward -= 700

          info["total_water"] = self.total_water
          info["final_yield"] = self.final_yield

      self.day += 1

      return obs, float(reward), terminated, truncated, info

    def _get_obs(self):
        th1 = float(self.model._init_cond.th[0])
        canopy = float(self.model._init_cond.canopy_cover)
        biomass = float(self.model._init_cond.biomass)

        th1_norm = np.clip(th1 / 0.45, 0.0, 1.0)
        canopy_norm = np.clip(canopy, 0.0, 1.0)
        biomass_norm = np.clip(biomass / 20000.0, 0.0, 1.0)

        weather_row = self.weather.iloc[self.day]

        tmin = float(weather_row["MinTemp"])
        tmax = float(weather_row["MaxTemp"])
        ref_et = float(weather_row["ReferenceET"])

        avg_temp = (tmin + tmax) / 2.0

        # temperature normalization
        temp_norm = np.clip((avg_temp + 5.0) / 50.0, 0.0, 1.0)

        # light proxy: AquaCrop weather does not directly give sunlight,
        # so ReferenceET is used as a proxy for heat/light demand
        light_norm = np.clip(ref_et / 8.0, 0.0, 1.0)

        # humidity proxy: high ET usually means drier air
        humidity_norm = np.clip(1.0 - (ref_et / 8.0), 0.0, 1.0)

        day_norm = np.clip(self.day / self.max_days, 0.0, 1.0)
        # soil temperature proxy
        soil_temp = avg_temp - 2.0
        soil_temp_norm = np.clip((soil_temp + 5.0) / 50.0, 0.0, 1.0)
        return np.array([day_norm,th1_norm, canopy_norm, biomass_norm, temp_norm,light_norm, humidity_norm,soil_temp_norm,], dtype=np.float32)

    def render(self):
        print(f"Day: {self.day}, Total water: {self.total_water}")

In [ ]:
env = AquaCropIrrigationEnv()
obs, info = env.reset()

for i in range(180):
    action = np.array([5.0], dtype=np.float32)
    obs, reward, terminated, truncated, info = env.step(action)

    print("day:", i)
    print("obs:", obs)
    print("reward:", reward)
    print("irr_cum:", env.model._init_cond.irr_cum)
    print("th1:", env.model._init_cond.th[0])
    print()

day: 0
obs: [0.        0.5835556 0.        0.        0.45      0.1875    0.8125
 0.41     ]
reward: -17.0
irr_cum: 30.0
th1: 0.2626

day: 1
obs: [0.00555556 0.8377778  0.         0.         0.33       0.1625
 0.8375     0.29      ]
reward: -30.0
irr_cum: 60.0
th1: 0.377

day: 2
obs: [0.01111111 0.84022224 0.         0.         0.25       0.15
 0.85       0.21      ]
reward: -30.0
irr_cum: 90.0
th1: 0.37810000000000005

day: 3
obs: [0.01666667 0.8231111  0.         0.         0.34       0.225
 0.775      0.3       ]
reward: -30.0
irr_cum: 120.0
th1: 0.3704

day: 4
obs: [0.02222222 0.8231111  0.         0.         0.37       0.175
 0.825      0.33      ]
reward: -30.0
irr_cum: 150.0
th1: 0.3704

day: 5
obs: [0.02777778 0.8377778  0.         0.         0.4        0.225
 0.775      0.36      ]
reward: -30.0
irr_cum: 180.0
th1: 0.377

day: 6
obs: [0.03333334 0.8426667  0.         0.         0.34       0.0875
 0.9125     0.3       ]
reward: -30.0
irr_cum: 210.0
th1: 0.37920000000000004

day:

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
env = DummyVecEnv([
    lambda: Monitor(AquaCropIrrigationEnv())
])

# модель будет видеть последние 4 состояния
env = VecFrameStack(env, n_stack=5)

policy_kwargs = dict(
    net_arch=dict(
        pi=[128, 128],
        vf=[128, 128]
    )
)

agent = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    n_steps=128,
    batch_size=64,
    learning_rate=0.0005,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs
)

agent.learn(total_timesteps=50000)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------
| time/              |     |
|    fps             | 184 |
|    iterations      | 1   |
|    time_elapsed    | 0   |
|    total_timesteps | 128 |
----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 180         |
|    ep_rew_mean          | -1.6e+03    |
| time/                   |             |
|    fps                  | 178         |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 256         |
| train/                  |             |
|    approx_kl            | 0.008359452 |
|    clip_fraction        | 0.0414      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.42       |
|    explained_variance   | -0.00597    |
|    learning_rate        | 0.0005      |
|    loss                 | 1.31e+04    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00934    |

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 180           |
|    ep_rew_mean          | -1.45e+03     |
| time/                   |               |
|    fps                  | 170           |
|    iterations           | 3             |
|    time_elapsed         | 2             |
|    total_timesteps      | 384           |
| train/                  |               |
|    approx_kl            | 0.00037063705 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.41         |
|    explained_variance   | -0.00826      |
|    learning_rate        | 0.0005        |
|    loss                 | 8.49e+03      |
|    n_updates            | 20            |
|    policy_gradient_loss | -0.000289     |
|    std                  | 0.991         |
|    value_loss           | 1.83e+04      |
-------------------------------------------


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 180          |
|    ep_rew_mean          | -1.45e+03    |
| time/                   |              |
|    fps                  | 161          |
|    iterations           | 160          |
|    time_elapsed         | 127          |
|    total_timesteps      | 20480        |
| train/                  |              |
|    approx_kl            | 0.0049260403 |
|    clip_fraction        | 0.0773       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.3         |
|    explained_variance   | -6.91e-06    |
|    learning_rate        | 0.0005       |
|    loss                 | 6.52e+03     |
|    n_updates            | 1590         |
|    policy_gradient_loss | -0.00688     |
|    std                  | 0.872        |
|    value_loss           | 1.24e+04     |
------------------------------------------
---------------------------------------
| rollout/    

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: Length of values (3) does not match length of index (243)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

env = DummyVecEnv([
    lambda: Monitor(AquaCropIrrigationEnv())
])

env = VecFrameStack(env, n_stack=5)

obs = env.reset()
done = False

total_reward = 0.0
actions = []

final_water = None
final_yield = None

while not done:
    action, _ = agent.predict(obs, deterministic=True)

    actions.append(float(action[0][0]))

    obs, reward, dones, infos = env.step(action)

    total_reward += float(reward[0])
    done = bool(dones[0])

    if done:
        print("Final info:", infos[0])
        final_water = infos[0].get("total_water")
        final_yield = infos[0].get("final_yield")

print("Total reward:", total_reward)
print("Total water:", final_water)
print("Total yield:", final_yield)
print("Min action:", min(actions))
print("Max action:", max(actions))
print("Avg action:", sum(actions) / len(actions))
print(actions)

Final info: {'total_water': 700.0, 'final_yield': 6.7059664739186475, 'episode': {'r': -1937.805116, 'l': 180, 't': 0.899508}, 'TimeLimit.truncated': False, 'terminal_observation': array([0.9722222 , 0.16195874, 0.726617  , 0.07560426, 0.64      ,
       0.8625    , 0.1375    , 0.6       , 0.9777778 , 0.16164327,
       0.70754695, 0.07571907, 0.64      , 0.8125    , 0.1875    ,
       0.6       , 0.98333335, 0.16119446, 0.68687284, 0.07582317,
       0.65      , 0.8       , 0.2       , 0.61      , 0.98888886,
       0.16065398, 0.6645471 , 0.07591535, 0.64      , 0.825     ,
       0.175     , 0.6       , 0.99444443, 0.1601144 , 0.64050347,
       0.07600079, 0.64      , 0.6875    , 0.3125    , 0.6       ],
      dtype=float32)}
Total reward: -1937.8051107525826
Total water: 700.0
Total yield: 6.7059664739186475
Min action: 0.23990632593631744
Max action: 1.0
Avg action: 0.9496521289977763
[0.23990632593631744, 0.37285324931144714, 0.48869818449020386, 0.6622291803359985, 0.7902593612

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
agent.save("ppo_aquacrop_irrigation_model(W_310)")

In [ ]:
from google.colab import files
from stable_baselines3 import PPO

env = AquaCropIrrigationEnv()
agent = PPO.load("ppo_aquacrop_irrigation_model(W_234)", env=env)
obs, info = env.reset()
done = False

total_reward = 0
actions = []

while not done:
    action, _ = agent.predict(obs, deterministic=True)
    actions.append(float(action[0]))

    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    done = terminated or truncated

print("Total reward:", total_reward)
print("Total water:", env.total_water)
print("Total yield:", env.final_yield)
print("Min action:", min(actions))
print("Max action:", max(actions))
print("Avg action:", sum(actions) / len(actions))
print(actions)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ValueError: Observation spaces do not match: Box(0.0, 1.0, (40,), float32) != Box(0.0, 1.0, (8,), float32)